<a href="https://colab.research.google.com/github/MDUBEY1998/Airbnb-bookings-analysis/blob/main/LLM_CHATBOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain==0.1.20 -q
!pip install -U langchain-community -q
!pip install langchain-huggingface -q
!pip install pypdf -q
!pip install faiss-cpu -q
!pip install ctransformers -q
!pip install -q \
    langchain==0.1.20 \
    langchain-core==0.1.52 \
    pydantic -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-checkpoint 4.1.0 requires langchain-core>=0.2.38, but you have langchain-core 0.1.53 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langch

In [8]:
!pip install numpy==2.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.0/19.0 MB 59.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.1.20 requires numpy<2,>=1, but you have numpy 2.0.0 which is incompatible.
langchain-community 0.0.38 requires numpy<2,>=1, but you have numpy 2.0.0 which is incompatible.
db-dtypes 1.6.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
bigquery-magics 0.14.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.


In [1]:

from langchain.chains import ConversationalRetrievalChain
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_community.vectorstores import FAISS  # Updated import
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.memory import ConversationBufferMemory
from langchain.llms import CTransformers
from transformers import AutoModel
from langchain.embeddings import HuggingFaceEmbeddings

In [3]:
# Load the PDF files from the path
loader = PyPDFLoader('/content/drive/MyDrive/llm_dataset/Fundamental of Physical Geography.pdf')
documents = loader.load()

In [4]:
# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
text_chunks = text_splitter.split_documents(documents)

# Create embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                   model_kwargs={'device': "cpu"})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
import gradio as gr
from langchain.vectorstores import FAISS
# from langchain.embeddings import HuggingFaceEmbeddings    # Replace with your actual embeddings class
# from langchain.memory import ConversationBufferMemory
# from langchain.chains import ConversationalRetrievalChain
# from langchain.llms import CTransformers  # Ensure you have CTransformers installed
#from your_module import text_chunks, embeddings  # Replace with your actual data sources

# Initialize vector store, LLM, and chain
vector_store = None
chain = None
setup_error = None

try:
    # Create the vector store from documents and embeddings
    vector_store = FAISS.from_documents(text_chunks, embeddings)
except IndexError as e:
    setup_error = f"An error occurred during vector store creation: {e}"
except Exception as e:
    setup_error = f"An unexpected error occurred: {e}"

if vector_store is not None:
    try:
        # Initialize the Language Model
        llm = CTransformers(
        model = "TheBloke/Llama-2-7B-Chat-GGUF",
        model_type="llama",
        config={'max_new_tokens': 128, 'temperature': 0.01}
        )

        # Initialize memory for conversation history
        memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

        # Create the Conversational Retrieval Chain
        chain = ConversationalRetrievalChain.from_llm(
            llm=llm,
            chain_type='stuff',
            retriever=vector_store.as_retriever(search_kwargs={"k": 2}),
            memory=memory
        )
    except Exception as e:
        setup_error = f"An error occurred during LLM or chain setup: {e}"

def conversation_chat(query, history):
    """
    Handles the conversation logic by querying the chain and updating history.

    Args:
        query (str): The user's input question.
        history (list): The chat history as a list of (user, bot) tuples.

    Returns:
        list: Updated chat history.
    """
    if setup_error:
        # If there was an error during setup, append it to the history
        history.append((query, setup_error))
        return history
    try:
        # Query the Conversational Retrieval Chain
        result = chain({"question": query, "chat_history": history})
        answer = result["answer"]
        # Append the user's query and the bot's response to the history
        history.append((query, answer))
    except Exception as e:
        # Handle any exceptions during the conversation
        history.append((query, f"An error occurred: {e}"))
    return history

def handle_submit(user_input, history):
    """
    Handles the submit action from the user.

    Args:
        user_input (str): The user's input question.
        history (list): The current chat history.

    Returns:
        tuple: Updated chat history for display and state.
    """
    if not user_input:
        # If the user input is empty, return the current history unchanged
        return history, history
    # Update the history with the new user input and bot response
    updated_history = conversation_chat(user_input, history)
    return updated_history, updated_history

# Define the Gradio interface
with gr.Blocks() as demo:
    # Title of the application
    gr.Markdown("# Your Mentor ")

    # Check if there was a setup error
    if setup_error:
        # Display the error message to the user
        gr.Markdown(f"**Error:** {setup_error}")
    else:
        # Initialize the Chatbot component to display the conversation
        chatbot = gr.Chatbot(label="Chat History")

        # Create a row for input components
        with gr.Row():
            with gr.Column():
                # Textbox for user to input questions
                user_input = gr.Textbox(
                    placeholder="Ask Your Doubt",
                    label="Question"
                )
                # Button to submit the question
                submit_button = gr.Button("Send")

        # State to keep track of chat history
        state = gr.State([])

        # Define the action when the submit button is clicked
        submit_button.click(
            handle_submit,
            inputs=[user_input, state],
            outputs=[chatbot, state]
        )

# Launch the Gradio interface
demo.launch()


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

/tmp/ipykernel_8407/2904841721.py:99: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat History")
/tmp/ipykernel_8407/2904841721.py:99: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat History")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3cd137bd3dbd80ba55.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
